# L7 demo: features, and the leak a scaler can hide

This notebook builds a small per-unit feature set for the NASA C-MAPSS turbofan
data (FD001), then predicts Remaining Useful Life (RUL) two ways: once with a
`StandardScaler` fit on the training engines only, and once with the same
scaler fit on training **and** test data combined, the "quick, obviously
fine" shortcut that is actually the single most common leakage bug in a
feature pipeline. We quantify the gap, and, just as importantly, look at why
the gap is not always dramatic, which is the more useful lesson than a scare
number.

## Run this first on Colab

Colab starts from its own preinstalled environment rather than this course's `uv`
environment, so run the cell below before anything else. It installs what this
notebook needs and Colab does not already have. Outside Colab it does nothing, so
you can run it or skip it.


In [ ]:
# Run this first on Colab. Anywhere else this cell does nothing.
#
# Only genuinely missing packages are installed, so Colab's own versions of
# everything it already ships are left alone.
#
# Generated by tools/colab_setup.py from this notebook's imports. Edit that.
import importlib.util
import subprocess
import sys

REQUIREMENTS = {
    "joblib": "joblib",
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
}


def _missing(module):
    try:
        return importlib.util.find_spec(module) is None
    except ModuleNotFoundError:  # the parent package is absent
        return True


if "google.colab" in sys.modules:
    need = sorted({pip for mod, pip in REQUIREMENTS.items() if _missing(mod)})
    if need:
        print("installing:", " ".join(need))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
    print("Colab setup done." if need else "Colab: nothing to install.")


## Get the data

FD001 is the smallest of the four C-MAPSS subsets: 100 training and 100 test
engines, one operating condition (sea level), one fault mode (high-pressure
compressor degradation). The training trajectories run to failure; the test
trajectories are truncated some unknown number of cycles before it, and
`RUL_FD001.txt` holds the true remaining cycles at each cutoff.

The cell below downloads and caches the data on first run. NASA's Prognostics
Center of Excellence has moved its repository more than once, and the current
landing page uses a JavaScript download widget rather than a stable file URL, so
this fetches from the S3 bucket that page serves the archive from. The download
is a zip containing another zip, which is worth knowing before you go looking
for the text files.

If the URL has moved again by the time you read this, download the "Turbofan
Engine Degradation Simulation Data Set" from the
[NASA PCoE Data Set Repository](https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/)
and drop `train_FD001.txt`, `test_FD001.txt`, and `RUL_FD001.txt` into
`lectures/l07/.cache/CMAPSS/` by hand. Everything below works either way, and
A4 uses the same files.

In [ ]:
import io
import urllib.request
import zipfile
from pathlib import Path

CACHE = Path('.cache/CMAPSS')
CACHE.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = CACHE / 'train_FD001.txt'
TEST_FILE = CACHE / 'test_FD001.txt'
RUL_FILE = CACHE / 'RUL_FD001.txt'
NEEDED = ['train_FD001.txt', 'test_FD001.txt', 'RUL_FD001.txt', 'readme.txt']

URL = ('https://phm-datasets.s3.amazonaws.com/NASA/'
       '6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip')

if not all((CACHE / f).exists() for f in NEEDED):
    print('downloading', URL)
    with urllib.request.urlopen(URL) as response:
        payload = response.read()

    outer = zipfile.ZipFile(io.BytesIO(payload))
    # The archive nests a second zip inside a directory. Find it rather than
    # hard-coding the path, because the directory name has spaces and a
    # leading "6. " that NASA has renamed before.
    inner_name = next(n for n in outer.namelist() if n.lower().endswith('.zip'))
    inner = zipfile.ZipFile(io.BytesIO(outer.read(inner_name)))
    for name in NEEDED:
        (CACHE / name).write_bytes(inner.read(name))
    print(f'cached {len(NEEDED)} files in {CACHE}/')

assert TRAIN_FILE.exists() and TEST_FILE.exists() and RUL_FILE.exists()
print((CACHE / 'readme.txt').read_text(errors='replace').split('Experimental')[0].strip()[:220])


## Stage 1: ingest

Each row is one engine, one cycle: a unit id, the cycle number, three
operational settings, and 21 sensor channels, whitespace separated with no
header. The test file is the same shape, except every unit's trajectory is
truncated **before** failure; `RUL_FD001.txt` holds the true remaining cycles
at that cutoff, one value per test unit, in unit order.

In [ ]:
import numpy as np
import pandas as pd

N_SETTINGS, N_SENSORS = 3, 21
COLUMNS = (
    ['unit', 'cycle']
    + [f'setting{i + 1}' for i in range(N_SETTINGS)]
    + [f'sensor{i + 1}' for i in range(N_SENSORS)]
)

def ingest(path):
    df = pd.read_csv(path, sep=r'\s+', header=None, names=COLUMNS)
    df[['unit', 'cycle']] = df[['unit', 'cycle']].astype(int)
    return df

train_raw = ingest(TRAIN_FILE)
test_raw = ingest(TEST_FILE)
rul_true = pd.read_csv(RUL_FILE, header=None, names=['rul'])
rul_true.index = np.arange(1, len(rul_true) + 1)   # unit ids are 1-indexed, in file order

print(train_raw.shape, 'train rows;', train_raw['unit'].nunique(), 'engines')
print(test_raw.shape, 'test rows;', test_raw['unit'].nunique(), 'engines')
train_raw.head(3)


## Stage 2: the RUL target

Train is run-to-failure, so an engine's RUL at any cycle is simply its final
cycle minus the current one. Most published C-MAPSS baselines **clip** this
at a ceiling (125 cycles here): an engine on cycle 3 of 300 is not
meaningfully "healthier" than one on cycle 30, since degradation only
becomes informative once it starts, and an unclipped linear target rewards a
model for memorizing each unit's total lifetime rather than reading the
degradation signal.

In [ ]:
MAX_RUL = 125

def add_rul(df, max_rul=MAX_RUL):
    df = df.copy()
    max_cycle = df.groupby('unit')['cycle'].transform('max')
    df['rul'] = (max_cycle - df['cycle']).clip(upper=max_rul)
    return df

train_raw = add_rul(train_raw)
train_raw[['unit', 'cycle', 'rul']].groupby('unit').tail(1).head(3)


## Stage 3: per-unit time-series features

Three feature families, computed **within each engine's own trajectory**
(`groupby('unit')`, never across engines): a rolling mean and standard
deviation, a delta from that engine's first recorded cycle, and the
cycle-to-cycle rate of change. All three read directly off the module's
Topics list, and all three would be silently wrong if computed before
grouping by unit, since cycle 1 of engine 12 has nothing to do with cycle 300
of engine 7.

Channel selection is not arbitrary. Of the 21 sensors in FD001, several are
**constant across all 20,631 training rows** and carry no information
whatsoever, exactly the situation the `clean` stage in [L5](../l05/notes.md)
exists to catch. The cell below checks for that before picking, and then takes
the three channels most strongly correlated with the target rather than
trusting a hard-coded list.

:::{admonition} Two bugs that were in this notebook
:class: warning

Both are worth seeing, because both ran without complaint and neither is
visible in the output.

The original version used `KEY_SENSORS = ['sensor2', 'sensor5', 'sensor8']`,
described in a comment as "the channels with a real degradation trend."
**`sensor5` is constant**: a single value, 14.62, in every training row. Its
rolling mean is that constant, its rolling standard deviation is zero, its
delta-from-first-cycle is zero, and its rate of change is zero. A third of the
"key" channels contributed four columns of nothing.

The features were then selected with
`[c for c in df.columns if any(c.startswith(s) for s in KEY_SENSORS)]`.
`'sensor20'` and `'sensor21'` both start with `'sensor2'`, so two unintended raw
channels were silently pulled into the feature matrix. They happen to be
*informative* channels, so the bug slightly **improved** the score, which is the
hardest kind of bug to notice: nothing looks wrong when a mistake helps. Select
columns by exact name, not by prefix, whenever your columns are numbered.
:::

In [ ]:
SENSORS = [f'sensor{i}' for i in range(1, N_SENSORS + 1)]

# Never carry a zero-variance channel into a feature pipeline. Test with
# nunique(), not std() == 0 -- see the note below, this matters here.
n_values = train_raw[SENSORS].nunique(dropna=True)
constant = [s for s in SENSORS if n_values[s] <= 1]
print(f'{len(constant)} of {N_SENSORS} sensors hold a single value: {constant}')

# How many would the "std is zero" test have caught?
spread = train_raw[SENSORS].std()
missed = {s: spread[s] for s in constant if spread[s] != 0}
print(f'  of those, std() == 0 exactly for {len(constant) - len(missed)}')
for s, v in missed.items():
    print(f'  {s}: one distinct value ({train_raw[s].iloc[0]}) but std() = {v:.3g}')

# Of what is left, take the three most strongly associated with the target.
informative = [s for s in SENSORS if s not in constant]
strength = train_raw[informative].corrwith(train_raw['rul']).abs().sort_values(ascending=False)
KEY_SENSORS = list(strength.index[:3])
print('\nstrongest remaining channels vs clipped RUL:')
print(strength.head(5).round(3).to_string())
print('using:', KEY_SENSORS)

WINDOW = 5

def engineer_features(df, sensors=KEY_SENSORS, window=WINDOW):
    """Add per-unit rolling, delta and rate-of-change features. Returns (df, names)."""
    df = df.sort_values(['unit', 'cycle']).copy()
    g = df.groupby('unit', group_keys=False)
    names = []
    for s in sensors:
        built = {
            f'{s}_roll_mean': lambda x: x.rolling(window, min_periods=1).mean(),
            f'{s}_roll_std': lambda x: x.rolling(window, min_periods=1).std().fillna(0),
            f'{s}_delta0': lambda x: x - x.iloc[0],
            f'{s}_roc': lambda x: x.diff().fillna(0),
        }
        for name, fn in built.items():
            df[name] = g[s].transform(fn)
        names.extend(built)
        names.append(s)          # the raw channel, by exact name
    return df, names

train_fe, feature_cols = engineer_features(train_raw)
test_fe, _ = engineer_features(test_raw)

# Exact-name selection, so 'sensor2' can never drag in 'sensor20'.
assert len(feature_cols) == 5 * len(KEY_SENSORS), feature_cols
assert train_fe[feature_cols].nunique().gt(1).all(), 'a constant feature survived'
print(f'\n{len(feature_cols)} features from {len(KEY_SENSORS)} channels, none constant')


## Stage 4a: the wrong way

Fit the scaler on training **and** test rows together, "because we have the
data anyway." This is exactly the bug the module flags as the most common
one in a feature pipeline: nothing here looks unusual, and nothing here
raises an error. The scaler simply learns statistics informed by data the
model will later be judged against.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

def last_cycle_per_unit(df):
    return df.sort_values('cycle').groupby('unit').tail(1).sort_values('unit')

last_rows = last_cycle_per_unit(test_fe)
y_true = rul_true.loc[last_rows['unit'], 'rul'].to_numpy()

# Two scalers: one honest, one leaky.
honest_scaler = StandardScaler().fit(train_fe[feature_cols])
combined = pd.concat([train_fe[feature_cols], test_fe[feature_cols]], axis=0)
leaky_scaler = StandardScaler().fit(combined)

# First question, before any model: did the leak actually change anything?
rel_mean = np.abs(honest_scaler.mean_ - leaky_scaler.mean_) / (np.abs(honest_scaler.mean_) + 1e-12)
rel_scale = np.abs(honest_scaler.scale_ - leaky_scaler.scale_) / honest_scaler.scale_
print('how much the two scalers disagree, across the 15 features:')
print(f'  centre: up to {rel_mean.max():.1%} relative difference')
print(f'  spread: up to {rel_scale.max():.1%}, median {np.median(rel_scale):.1%}')
print('\nThe leak is real and measurable in the transform itself.')

leaky_model = Ridge(alpha=10.0).fit(
    leaky_scaler.transform(train_fe[feature_cols]), train_fe['rul']
)
pred_leaky = leaky_model.predict(leaky_scaler.transform(last_rows[feature_cols]))
rmse_leaky = mean_squared_error(y_true, pred_leaky) ** 0.5
print(f'\nleaky-scaler RMSE:  {rmse_leaky:.3f} RUL-cycles')


## Stage 4b: the right way, as an `sklearn` `Pipeline`

Wrapping the scaler and the model in a single `Pipeline` makes the leak
structurally harder to write by accident: `pipeline.fit(X_train, y_train)`
has no way to see `X_test`, so there is no line of code left where "fit on
everything" could sneak back in.

In [ ]:
from sklearn.pipeline import Pipeline

correct_pipeline = Pipeline([
    ('scale', StandardScaler()),
    ('model', Ridge(alpha=10.0)),
]).fit(train_fe[feature_cols], train_fe['rul'])

pred_correct = correct_pipeline.predict(last_rows[feature_cols])
rmse_correct = mean_squared_error(y_true, pred_correct) ** 0.5
print(f'correct-pipeline RMSE: {rmse_correct:.3f} RUL-cycles')
print(f'leaky-scaler RMSE:     {rmse_leaky:.3f} RUL-cycles')
print(f'gap:                   {rmse_leaky - rmse_correct:+.3f} RUL-cycles')


## Reading the gap honestly

The gap is essentially zero: a thousandth of a cycle on an RMSE of about
nineteen. Meanwhile the cell above showed the two scalers disagreeing by up to
23% on where a feature is centred. Both of those things are true at once.

**The leak happened; the metric did not report it.**

Your validation score is not a leak detector. It is one scalar summary of one
model's behaviour on one dataset, and it can be completely insensitive to a
transform having been fitted on data it should never have seen.

Why is `Ridge` insensitive here? Because standardizing inputs mostly rescales
the coefficient the model learns, and Ridge compensates almost perfectly at this
penalty strength. The cell below makes that concrete by running the identical
leak past three models that differ only in how much they care about scale.
Predict the ordering before you run it.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor

def rmse_under(scaler, make_model):
    """Fit make_model() on scaler-transformed train, score on the test cutoffs."""
    model = make_model().fit(scaler.transform(train_fe[feature_cols]), train_fe['rul'])
    pred = model.predict(scaler.transform(last_rows[feature_cols]))
    return mean_squared_error(y_true, pred) ** 0.5

CANDIDATES = [
    ('LinearRegression', LinearRegression, 'exactly scale-invariant'),
    ('Ridge(alpha=10)', lambda: Ridge(alpha=10.0), 'penalty barely bites'),
    ('Ridge(alpha=1e4)', lambda: Ridge(alpha=1e4), 'penalty dominates'),
    ('KNeighbors(k=5)', lambda: KNeighborsRegressor(5), 'distances are all scale'),
]

print(f'{"model":18s} {"honest":>8s} {"leaky":>8s} {"gap":>9s}   why')
print('-' * 68)
for name, factory, why in CANDIDATES:
    honest = rmse_under(honest_scaler, factory)
    leaky = rmse_under(leaky_scaler, factory)
    print(f'{name:18s} {honest:8.3f} {leaky:8.3f} {leaky - honest:+9.4f}   {why}')


Four models, one leak, four different verdicts. `LinearRegression` reports the
gap as exactly zero, because a least-squares fit is genuinely invariant to
rescaling its inputs: whatever the scaler does, the coefficients undo. Ridge at a
mild penalty is nearly as blind. Push the penalty up and the leak becomes
visible, though it can land in the "wrong" direction, making the leaky pipeline
look *better*. A leak makes your score meaningless rather than reliably
inflating it. And a nearest-neighbours model, where every prediction is a
distance computation and distances are nothing but scale, shows a gap you could
not miss.

So the size of a measured leak tells you about the model you happened to
measure with. It tells you nothing about whether the leak was acceptable.

None of that is a reason to relax the rule. It is the reason the rule has to be
**structural** rather than case-by-case: you cannot know in advance whether this
month's data, this quarter's model, or next year's dataset is the one where the
leak is large. The `Pipeline` below removes that choice entirely.

The much larger and easier way to leak on this exact dataset, letting a random
row-level split scatter one engine's cycles across both train and test, is
deliberately not shown here. That is L8's leakage taxonomy, next session, and it
is the one that produces the dramatic inflated metric.

## Persisting the fitted pipeline

The whole point of fitting inside a `Pipeline` is that the fitted object,
scaler statistics and all, is one artifact you can save, version, and reload
exactly, rather than a script you have to rerun and hope reproduces the same
numbers.

In [ ]:
import joblib

MODEL_PATH = CACHE.parent / 'rul_pipeline.joblib'
joblib.dump(correct_pipeline, MODEL_PATH)

reloaded = joblib.load(MODEL_PATH)
check = reloaded.predict(last_rows[feature_cols])
assert np.allclose(check, pred_correct)
print('reloaded pipeline reproduces the saved predictions exactly:', MODEL_PATH)


Full notes, with the physical-feature and spectral-feature material this
notebook does not cover: [`../notes.md`](notes.md).